# ISL0661 capture rate: why, not just how much

Annual capture rates come from official Electricity Authority half-hourly final
prices at ISL0661. The production shape is modelled, not measured. This
notebook takes the headline apart into the two questions that actually get
asked: is the output in the expensive **months**, and is it in the expensive
**hours**?

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from nz_solar_siting.capture import yearly_capture_rates
from nz_solar_siting.solar_shape import half_hour_shape

prices = pd.read_csv(ROOT / "data" / "derived" / "ISL0661_2019_2025.csv.gz")
load = pd.read_csv(ROOT / "data" / "derived" / "ISL0661_load_2019_2025.csv.gz")
rates = yearly_capture_rates(prices, tilt_deg=25.0, load_shape=load)
rates.round(4)

## The additive split

Writing each half hour's price as a daily mean plus a within-day residual makes
the capture rate separate exactly into `seasonal_term + intraday_term`. The
seasonal term is what the same daily energy would capture with a flat
within-day profile. The intraday term is everything the shape of the day adds.
Only the second one is midday price cannibalisation.

In [ ]:
split = rates[["year", "solar_capture_rate", "seasonal_capture_rate", "intraday_capture_points"]].copy()
split["recombined"] = split["seasonal_capture_rate"] + split["intraday_capture_points"]
split["seasonal_share_of_gap"] = (
    (1 - split["seasonal_capture_rate"]) / (1 - split["solar_capture_rate"])
).round(3)
split.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(rates["year"] - 0.18, (rates["seasonal_capture_rate"] - 1) * 100, width=0.34,
       color="#2d6a9f", label="Seasonal term")
ax.bar(rates["year"] + 0.18, rates["intraday_capture_points"] * 100, width=0.34,
       color="#e5a300", label="Intraday term")
ax.axhline(0, color="#101820", linewidth=1)
ax.set(xlabel="NZ market year", ylabel="Percentage points", title="Where the gap comes from")
ax.set_xticks(rates["year"].tolist())
ax.legend(frameon=False)
ax.grid(alpha=0.2, axis="y")
fig.tight_layout()

## The worst year

The monthly picture explains it: winter prices were extreme during the dry-year
squeeze, and that is exactly when a fixed-tilt array in Canterbury produces
least. The intraday profile of the same year has no pronounced midday trough.

In [ ]:
local = pd.to_datetime(prices["timestamp_utc"], utc=True).dt.tz_convert("Pacific/Auckland")
year = int(rates.loc[rates["solar_capture_rate"].idxmin(), "year"])
subset = prices.loc[local.dt.year == year]
subset_local = local.loc[subset.index]

shape = half_hour_shape(year, tilt_deg=25.0)
shape_local = shape["midpoint_utc"].dt.tz_convert("Pacific/Auckland")

monthly = pd.DataFrame({
    "mean_price_nzd_mwh": subset.groupby(subset_local.dt.month)["price_nzd_mwh"].mean(),
    "output_share_pct": 100 * shape.groupby(shape_local.dt.month)["output_pu"].sum() / shape["output_pu"].sum(),
})
monthly.index.name = "month"
monthly.round(1)

In [ ]:
hourly = pd.DataFrame({
    "mean_price_nzd_mwh": subset.groupby(subset_local.dt.hour)["price_nzd_mwh"].mean(),
    "output_share_pct": 100 * shape.groupby(shape_local.dt.hour)["output_pu"].sum() / shape["output_pu"].sum(),
})
hourly.index.name = "hour"
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
for axis, frame, label in ((left, monthly, "Month"), (right, hourly, "Hour of day")):
    axis.bar(frame.index, frame["mean_price_nzd_mwh"], color="#2d6a9f")
    axis.set(xlabel=f"{label} ({year})", ylabel="Mean price (NZD/MWh)")
    twin = axis.twinx()
    twin.plot(frame.index, frame["output_share_pct"], marker="o", color="#e5a300")
    twin.set_ylabel("Modelled output share (%)")
fig.suptitle(f"Seasonal mismatch is large in {year}; the intraday mismatch is not")
fig.tight_layout()

## Control group: the real load shape at the same node

A flat profile is an arithmetic invariant, not a comparison. The Electricity
Authority publishes metered grid-exit energy at ISL0661, so the local demand
shape runs through the identical calculation. It peaks on winter evenings and
captures above 1.0 in every year - the mirror image of solar, and the reason
"generation is not revenue" is a statement about timing rather than about PV.

In [ ]:
control = rates[[
    "year", "solar_capture_rate", "load_capture_rate", "flat_capture_rate",
    "load_seasonal_capture_rate", "load_intraday_capture_points",
]]
print("load capture above 1.0 in every year:", bool((control["load_capture_rate"] > 1).all()))
control.round(4)

## Sensitivity: tilt is not a cosmetic assumption

The array is modelled as north-facing at a configured tilt. A horizontal plane
(0 degrees) overstates the summer-to-winter swing badly, and because this node
prices winter so highly, that error pushes the capture rate down. This is the
sensitivity that matters most to the headline number.

In [ ]:
tilt_rows = []
for tilt in (0.0, 25.0, 35.0):
    result = yearly_capture_rates(prices, tilt_deg=tilt)
    shape = half_hour_shape(2024, tilt_deg=tilt)
    month = shape["midpoint_utc"].dt.tz_convert("Pacific/Auckland").dt.month
    monthly_output = shape.groupby(month)["output_pu"].sum()
    tilt_rows.append({
        "tilt_deg": tilt,
        "december_over_june_output": monthly_output.loc[12] / monthly_output.loc[6],
        "mean_capture_rate": result["solar_capture_rate"].mean(),
        "minimum_capture_rate": result["solar_capture_rate"].min(),
    })
pd.DataFrame(tilt_rows).round(4)

## What this is not

Nodal spot prices, not a PPA. No loss factors, basis, FTRs, hedges, curtailment
or dispatch constraints. A clear-sky timing proxy, not measured generation: no
cloud, snow, soiling, degradation, tracking or inverter clipping. The intraday
term will also grow more negative as PV penetration rises; the near-zero values
here describe a market that has very little solar in it yet.